# KNN Model

- BigQuery ML does not have a native model_type = 'KNN_CLASSIFIER' because KNN is a lazy learner (it stores all data points in memory and computes mathematical distances on the fly rather than training weights).

- In BigQuery, you have two standard ways to work with the iris.csv dataset:
    1. **Method 1**: Pure SQL K-Nearest Neighbors (Exact Distance Calculation) — Uses Euclidean distance in SQL to find the top K closest neighbors and take a majority vote (the exact mathematical definition of KNN).
    2. **Method 2**: Multi-Class BQML Classifier (LOGISTIC_REG / Softmax) — The native BQML way to classify Iris into 3 species (setosa, versicolor, virginica).
    3. **Method 3**: K-Means Clustering (KMEANS, K=3) — If you want unsupervised neighbor-based clustering.

# KNN Model Step by step Complete Guide:- 

- **Step 1**: Create Schema / Dataset

```sql
-- Create a schema/dataset in the US region
CREATE SCHEMA IF NOT EXISTS `gcp-aiml-sandbox-rahuld-1234.student_sandbox`
OPTIONS(
  location = 'US'
);
```
> gcp-aiml-sandbox-rahuld-1234.student_sandbox.raw_iris

---

- **Step 2**: Cleanse and Preprocess Data -> We standardize column names, add an ID, and ensure all dimensions are FLOAT64.

```sql
-- Cleanse data: assign unique IDs and ensure numeric types
CREATE OR REPLACE TABLE `gcp-aiml-sandbox-rahuld-1234.student_sandbox.cleaned_iris` AS
SELECT
  ROW_NUMBER() OVER() AS flower_id,
  CAST(sepal_length AS FLOAT64) AS sepal_length,
  CAST(sepal_width AS FLOAT64) AS sepal_width,
  CAST(petal_length AS FLOAT64) AS petal_length,
  CAST(petal_width AS FLOAT64) AS petal_width,
  -- Ensure target is clean string (setosa, versicolor, virginica)
  CAST(species AS STRING) AS species
FROM
  `gcp-aiml-sandbox-rahuld-1234.student_sandbox.raw_iris`;
```

---

- **Method 1**: Pure K-Nearest Neighbors (KNN, K=3) in BigQuery SQL -> This calculates the Euclidean distance 
from an unknown flower to all known flowers in the database, grabs the 3 closest neighbors, and votes on the species.

```sql
-- Test flower we want to classify
WITH unseen_flower AS (
  SELECT 
    'Unknown_Flower_1' AS sample_name,
    5.1 AS sepal_length,
    3.5 AS sepal_width,
    1.4 AS petal_length,
    0.2 AS petal_width
),

-- Step 1: Compute Euclidean Distance to every known flower
distances AS (
  SELECT
    u.sample_name,
    k.species,
    -- Euclidean distance formula in 4D space
    SQRT(
      POW(u.sepal_length - k.sepal_length, 2) +
      POW(u.sepal_width - k.sepal_width, 2) +
      POW(u.petal_length - k.petal_length, 2) +
      POW(u.petal_width - k.petal_width, 2)
    ) AS euclidean_distance
  FROM
    unseen_flower u
  CROSS JOIN
    `gcp-aiml-sandbox-rahuld-1234.student_sandbox.cleaned_iris` k
),

-- Step 2: Rank the neighbors (Find K=3 Nearest Neighbors)
ranked_neighbors AS (
  SELECT
    sample_name,
    species,
    euclidean_distance,
    ROW_NUMBER() OVER (PARTITION BY sample_name ORDER BY euclidean_distance ASC) AS neighbor_rank
  FROM
    distances
)

-- Step 3: Majority Vote across K=3 nearest neighbors
SELECT
  sample_name,
  species AS predicted_species,
  COUNT(*) AS votes_out_of_3
FROM
  ranked_neighbors
WHERE
  neighbor_rank <= 3 -- K = 3
GROUP BY
  sample_name, species
ORDER BY
  votes_out_of_3 DESC
LIMIT 1;
```

---

- **Method 2**: BQML Multi-Class Classifier (Native Supervised Model) -> Because the Iris dataset has 3 categories (setosa, versicolor, virginica), BigQuery ML automatically switches to Multinomial Logistic Regression (Softmax). Train the Multi-Class Model:

```sql
CREATE OR REPLACE MODEL `gcp-aiml-sandbox-rahuld-1234.student_sandbox.iris_multiclass_model`
OPTIONS (
  model_type = 'LOGISTIC_REG',
  input_label_cols = ['species'],
  data_split_method = 'RANDOM',
  data_split_eval_fraction = 0.2
) AS
SELECT
  sepal_length,
  sepal_width,
  petal_length,
  petal_width,
  species
FROM
  `gcp-aiml-sandbox-rahuld-1234.student_sandbox.cleaned_iris`;
```

- Evaluate Multi-Class Performance:

```sql 
SELECT * FROM ML.EVALUATE(MODEL `gcp-aiml-sandbox-rahuld-1234.student_sandbox.iris_multiclass_model`);

```

- Predict with Probabilities for all 3 Classes:
```sql
SELECT
  predicted_species,
  predicted_species_probs,
  sepal_length,
  sepal_width,
  petal_length,
  petal_width
FROM
  ML.PREDICT(
    MODEL `gcp-aiml-sandbox-rahuld-1234.student_sandbox.iris_multiclass_model`,
    (
      SELECT 5.1 AS sepal_length, 3.5 AS sepal_width, 1.4 AS petal_length, 0.2 AS petal_width
      UNION ALL
      SELECT 6.7 AS sepal_length, 3.0 AS sepal_width, 5.2 AS petal_length, 2.3 AS petal_width
    )
  );
```

---


- **Method 3**: Unsupervised K-Means Clustering (K=3) -> 
If your assignment was looking for cluster-based grouping with K=3: 
- Train K-Means Model:

```sql
CREATE OR REPLACE MODEL `gcp-aiml-sandbox-rahuld-1234.student_sandbox.iris_kmeans_model`
OPTIONS (
  model_type = 'KMEANS',
  num_clusters = 3,
  standardize_features = TRUE
) AS
SELECT
  sepal_length,
  sepal_width,
  petal_length,
  petal_width
FROM
  `gcp-aiml-sandbox-rahuld-1234.student_sandbox.cleaned_iris`;
```

---


- **View Cluster Centers (Centroids)**:
```sql
SELECT
  *
FROM
  ML.CENTROIDS(MODEL `gcp-aiml-sandbox-rahuld-1234.student_sandbox.iris_kmeans_model`);
```

---


- **Step 8**: Clean Up All Resources (Optional)

```sql
DROP MODEL IF EXISTS `gcp-aiml-sandbox-rahuld-1234.student_sandbox.iris_multiclass_model`;
DROP MODEL IF EXISTS `gcp-aiml-sandbox-rahuld-1234.student_sandbox.iris_kmeans_model`;
DROP TABLE IF EXISTS `gcp-aiml-sandbox-rahuld-1234.student_sandbox.cleaned_iris`;
DROP TABLE IF EXISTS `gcp-aiml-sandbox-rahuld-1234.student_sandbox.raw_iris`;
DROP SCHEMA IF EXISTS `gcp-aiml-sandbox-rahuld-1234.student_sandbox` CASCADE;
```

---